In [ ]:
from pathlib import Path
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive

# client_secrets.json lives in the repo root, not next to this notebook
secrets = next(p / "client_secrets.json" for p in [Path.cwd(), *Path.cwd().parents] if (p / "client_secrets.json").exists())

gauth = GoogleAuth()
gauth.settings["client_config_file"] = str(secrets)
gauth.settings["get_refresh_token"] = True  # lets the saved login be renewed silently
creds_file = str(secrets.parent / "credentials.json")

gauth.LoadCredentialsFile(creds_file)  # reuse saved login if present
if gauth.credentials is None:
    gauth.LocalWebserverAuth()  # first time only: opens browser
elif gauth.access_token_expired:
    gauth.Refresh()
else:
    gauth.Authorize()
gauth.SaveCredentialsFile(creds_file)

drive = GoogleDrive(gauth)

file_list = drive.ListFile({
    "q": "trashed=false"
}).GetList()

for file in file_list:
    print(file["title"], file["id"], file["mimeType"])

In [ ]:
# Google Docs -> export as plain text
doc = drive.CreateFile({"id": "1PxjgLTC9XGogl4694tiM8Zv3eUkXqWDWZbUfLw3IZ48"})
print(doc.GetContentString(mimetype="text/plain"))

# Markdown file -> read as-is
md = drive.CreateFile({"id": "1zCXQD0MRVAvo3QbIlDgc6gijCxePKfCe"})
print(md.GetContentString())